In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers sentencepiece huggingface_hub accelerate

In [3]:
from pathlib import Path
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

In [4]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ml_projects/NLP_Text_Summarization"
)
SRC_DIR = PROJECT_ROOT / "src"
CONFIG_DIR = PROJECT_ROOT / "config"
SRC_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using device:", device)

Using device: cpu


In [8]:
MODEL_NAME = "hyperchancellor07/pegasus-samsum-dialogue-summarizer"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
)
model.to(device)
print("Fine-tuned PEGASUS loaded successfully.")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Fine-tuned PEGASUS loaded successfully.


In [9]:
def generate_summary(dialogue_text):

    dialogue_text = "summarize: " + dialogue_text
    inputs = tokenizer(
        dialogue_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    summary_ids = model.generate(
        **inputs,
        max_length=64,
        min_length=8,
        num_beams=8,
        repetition_penalty=2.5,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        early_stopping=True,
    )
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
    )
    return generated_summary

In [10]:
sample_dialogue = """
Amanda: I baked cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)
"""
generated_summary = generate_summary(sample_dialogue)
print("Dialogue:\n")
print(sample_dialogue)
print("Generated Summary:\n")
print(generated_summary)

Dialogue:


Amanda: I baked cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Generated Summary:

Amanda baked cookies and will bring them to Jerry tomorrow.


In [11]:
prediction_pipeline_code = '''
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)


MODEL_NAME = "your-username/pegasus-samsum-full-finetuned"


class DialogueSummarizer:

    def __init__(self):

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )


        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME
        )


        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL_NAME,
            use_safetensors=False,
        )


        self.model.to(self.device)


    def summarize(self, dialogue_text):
        dialogue_text = "summarize: " + dialogue_text


        inputs = self.tokenizer(
            dialogue_text,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512,
        )


        inputs = {
            key: value.to(self.device)
            for key, value in inputs.items()
        }


        summary_ids = self.model.generate(
            **inputs,

            max_length=64,
            min_length=8,

            num_beams=8,

            repetition_penalty=2.5,
            no_repeat_ngram_size=3,

            length_penalty=1.0,

            early_stopping=True,
        )


        generated_summary = self.tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
        )


        return generated_summary
'''


In [12]:
prediction_pipeline_path = (
    SRC_DIR / "prediction_pipeline.py"
)
with open(prediction_pipeline_path, "w") as f:
    f.write(prediction_pipeline_code)
    print(f"Saved: {prediction_pipeline_path}")

Saved: /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/src/prediction_pipeline.py


In [13]:
streamlit_app_code = '''
import streamlit as st

from src.prediction_pipeline import DialogueSummarizer


st.set_page_config(
    page_title="Dialogue Summarization System",
    page_icon="🧠",
    layout="wide",
)


@st.cache_resource

def load_model():
    return DialogueSummarizer()


summarizer = load_model()


st.markdown(
    """
    <style>
    .main {
        padding-top: 1rem;
    }

    .title-text {
        font-size: 42px;
        font-weight: 700;
        color: #4F46E5;
        margin-bottom: 0.2rem;
    }

    .subtitle-text {
        font-size: 18px;
        color: #6B7280;
        margin-bottom: 2rem;
    }

    .stTextArea textarea {
        border-radius: 12px;
        font-size: 16px;
    }

    .summary-box {
        background-color: #F3F4F6;
        padding: 1.2rem;
        border-radius: 14px;
        border: 1px solid #E5E7EB;
        font-size: 17px;
        line-height: 1.7;
        color: #111827;
    }
    </style>
    """,
    unsafe_allow_html=True,
)


st.markdown(
    '<div class="title-text">🧠 Dialogue Summarization System</div>',
    unsafe_allow_html=True,
)

st.markdown(
    '<div class="subtitle-text">Transformer-based conversational summarization using PEGASUS.</div>',
    unsafe_allow_html=True,
)


col1, col2 = st.columns([1.2, 1])


with col1:

    user_input = st.text_area(
        "💬 Enter Conversation",
        placeholder="Paste conversation here...",
        height=420,
    )

'''
app_file_path = PROJECT_ROOT / "app.py"
with open(app_file_path, "w") as f:
    f.write(streamlit_app_code)
print(f"Saved: {app_file_path}")

Saved: /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/app.py


In [14]:
readme_text = '''
# NLP Dialogue Summarization System

## Overview

This project performs dialogue summarization using:
- PEGASUS transformer
- SAMSum dataset
- Fine-tuning and PEFT adaptation
- Streamlit deployment


## Features

- Dialogue summarization
- Transformer-based inference
- Hugging Face integration
- Streamlit frontend


## Tech Stack

- Python
- Transformers
- Hugging Face
- Streamlit
- PyTorch


## Run Streamlit App

```bash
streamlit run app.py
'''
readme_path = PROJECT_ROOT / "README.md"
with open(readme_path, "w") as f:
    f.write(readme_text)

In [15]:
%cd /content/drive/MyDrive/ml_projects/NLP_Text_Summarization

/content/drive/MyDrive/ml_projects/NLP_Text_Summarization


In [16]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/.git/


In [17]:
!git config --global user.name "hyperchancellor07"
!git config --global user.email "yadavkirtan44@gmail.com"

In [18]:
!git remote add origin https://github.com/hyperchancellor07/NLP-Dialogue-Summarization

In [19]:
%%writefile .gitignore

artifacts/models/
*.pt
*.bin
*.ckpt
__pycache__/
.ipynb_checkpoints/

Writing .gitignore


In [ ]:
!git add .